# Online (Prolific) Cohort Preprocessing

| Field | Value |
|---|---|
| **Manuscript panels / stats** | Methods: Online cohort exclusions (N=208 -> 200 decision-making -> 193 recognition memory) |
| **Inputs** | `$MEMORYBANDIT_DATA_DIR/projects/guLab/Salman/Prolific/DelayedMemoryTest/*.csv`, `demographics.csv` |
| **Outputs** | `learn_df_prolific_v2.csv`, `full_data_df_prolific_v2.csv` |
| **Dependencies** | `membandit`, `numpy`, `pandas`, `scipy`, `scikit-learn` |
| **Expected runtime** | ~15 seconds |
| **Upstream steps** | None (upstream of online behavior & modeling notebooks) |


# Prolific Cohort Behavioral Preprocessing

Builds the trial-level learning (`learn_df_prolific_v2.csv`) and recognition memory (`full_data_df_prolific_v2.csv`) tables from the raw Prolific task exports and computes participant-level decision-making and recognition sensitivity ($d'$) flags.


## Setup


In [ ]:
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.metrics import auc, roc_curve

from membandit import paths
BASE_DIR = Path(paths.get_base_dir())
RAW_DIR = BASE_DIR / "projects/guLab/Salman/Prolific/DelayedMemoryTest"
SAVE_DIR = Path(paths.get_prolific_dir())

NTRIALS = 80
REWARD_THRESHOLD = 40  # total points > threshold → good_dm_performance

print({"RAW_DIR": str(RAW_DIR), "SAVE_DIR": str(SAVE_DIR), "exists": RAW_DIR.exists()})


## Helpers

Self-contained `dprime` (Hautus-style edge correction, matching `ImgRatingSorter/util.py`). Good-DM is flagged from `learn_df` reward sums only.


In [ ]:
def dprime(pHit, pFA, PresentT, AbsentT, criteria=False):
    """Signal-detection d' with edge correction for perfect hit / zero FA rates."""
    if pHit == 1:
        pHit = 1 - (1 / (2 * PresentT))
    if pFA == 0:
        pFA = 1 / (2 * AbsentT)
    zHit = norm.ppf(pHit)
    zFA = norm.ppf(pFA)
    dPrime = zHit - zFA
    if criteria:
        beta = np.exp((zFA**2 - zHit**2) / 2)
        C = -0.5 * (zHit + zFA)
        return dPrime, beta, C
    return dPrime


def matthews_corrcoef(tp, tn, fp, fn):
    numerator = (tp * tn) - (fp * fn)
    denominator = ((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) ** 0.5
    if denominator == 0:
        return 0
    return numerator / denominator


def compute_recognition_sdt(df: pd.DataFrame) -> pd.DataFrame:
    """Per-participant hit rate, FA rate, d', criterion, MCC, AUC."""
    df = df.copy()
    for pid in df.participant.unique():
        subj_df = df[df.participant == pid]
        hits = subj_df[(subj_df._merge == "both") & (subj_df.recog_choice_bool == 1)].shape[0]
        misses = subj_df[(subj_df._merge == "both") & (subj_df.recog_choice_bool == 0)].shape[0]
        false_alarms = subj_df[(subj_df._merge == "left_only") & (subj_df.recog_choice_bool == 1)].shape[0]
        correct_rejections = subj_df[(subj_df._merge == "left_only") & (subj_df.recog_choice_bool == 0)].shape[0]

        hm = hits + misses
        fc = false_alarms + correct_rejections
        hrs = hits / hm if hm else np.nan
        fars = false_alarms / fc if fc else np.nan

        df.loc[df.participant == pid, "hit_rate"] = hrs
        df.loc[df.participant == pid, "false_alarm_rate"] = fars
        df.loc[df.participant == pid, "dprime"] = np.nan
        df.loc[df.participant == pid, "crit"] = np.nan
        df.loc[df.participant == pid, "auc"] = np.nan
        df.loc[df.participant == pid, "mcc"] = np.nan

        if not (np.isfinite(hrs) and np.isfinite(fars) and hm > 0 and fc > 0):
            continue

        dp, _, crit = dprime(hrs, fars, hm, fc, criteria=True)
        mcc = matthews_corrcoef(hits, correct_rejections, false_alarms, misses)

        signal = subj_df[(subj_df.hits == 1) | (subj_df.misses == 1)].recog_choice_bool
        noise = subj_df[(subj_df.correct_rejections == 1) | (subj_df.false_alarms == 1)].recog_choice_bool
        n_samples = len(signal)
        responses = np.concatenate((signal, noise))
        true_labels = np.concatenate((np.ones(n_samples), np.zeros(n_samples)))
        fpr, tpr, _ = roc_curve(true_labels, responses)
        roc_auc = auc(fpr, tpr)

        if dp != float("-inf"):
            df.loc[df.participant == pid, "dprime"] = dp
            df.loc[df.participant == pid, "crit"] = crit
            df.loc[df.participant == pid, "mcc"] = mcc
            df.loc[df.participant == pid, "auc"] = roc_auc
    return df


def attach_image_dprime(df: pd.DataFrame, all_mem_df: pd.DataFrame) -> pd.DataFrame:
    """Merge normative image memorability (DPRIME → image_dprime) by img_path."""
    return df.merge(all_mem_df, on="img_path", how="left").rename(columns={"DPRIME": "image_dprime"})


def flag_good_dm_from_learn(learn_df: pd.DataFrame, reward_threshold: float = REWARD_THRESHOLD) -> pd.Series:
    """Return participant → bool Series from *learning* reward totals only."""
    totals = learn_df.groupby("participant")["reward"].sum()
    return (totals > reward_threshold).rename("good_dm_performance")


def attach_good_dm_flag(df: pd.DataFrame, good_map: pd.Series) -> pd.DataFrame:
    out = df.copy()
    out["good_dm_performance"] = out["participant"].map(good_map)
    return out


def add_recognition_outcomes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["recog_choice_bool"] = df.recog_choice.apply(lambda x: 1 if x == 2.0 else 0)
    df["recog_choice"] = df.recog_choice.apply(lambda x: "old" if x == 2.0 else "new")
    df["hits"] = (df._merge == "both") & (df.recog_choice_bool == 1)
    df["misses"] = (df._merge == "both") & (df.recog_choice_bool == 0)
    df["false_alarms"] = (df._merge == "left_only") & (df.recog_choice_bool == 1)
    df["correct_rejections"] = (df._merge == "left_only") & (df.recog_choice_bool == 0)
    return df


print("helpers ready")


## Load demographics, Day 1 / Day 2 databases, image memorability


In [ ]:
demo_file = RAW_DIR / "demographics.csv"
demographics_df = pd.read_csv(demo_file, index_col=False, header=0)
# Some Prolific demographic rows are column-shifted
shifted = ~demographics_df["Unnamed: 19"].isna()
demographics_df.loc[shifted, "Age"] = demographics_df.loc[shifted, "Sex"]
demographics_df.loc[shifted, "Sex"] = demographics_df.loc[shifted, "Ethnicity simplified"]
demographics_df = demographics_df[["Participant id", "Age", "Sex"]].rename(
    columns={"Participant id": "participant", "Age": "age", "Sex": "sex"}
)
demo_ids = demographics_df.participant.unique().tolist()

day1_files = sorted(glob(str(RAW_DIR / "Day1/db" / "*.csv")))
day2_files = sorted(glob(str(RAW_DIR / "Day2/db" / "*.csv")))
assert day1_files and day2_files, "missing Day1/Day2 db CSVs"
day1_df = pd.read_csv(day1_files[0], index_col=None, header=0, low_memory=False)
day2_df = pd.read_csv(day2_files[0], index_col=None, header=0, low_memory=False)
day1_df = day1_df[day1_df.participant.isin(demo_ids)].reset_index(drop=True)
day2_df = day2_df[day2_df.participant.isin(demo_ids)].reset_index(drop=True)

# Prefer CSV (no openpyxl); fall back to xlsx next to raw data / EMU path
mem_csv = RAW_DIR / "all_mem_data_long.csv"
mem_xlsx = RAW_DIR / "all_mem_data_long.xlsx"
ws_csv = Path("..") / "data" / "all_mem_data_long.csv"
if mem_csv.exists():
    all_mem_df = pd.read_csv(mem_csv)
elif ws_csv.exists():
    all_mem_df = pd.read_csv(ws_csv)
elif mem_xlsx.exists():
    all_mem_df = pd.read_excel(mem_xlsx, engine="openpyxl")
else:
    raise FileNotFoundError("all_mem_data_long.{csv,xlsx} not found")
all_mem_df = all_mem_df[["img_path", "DPRIME"]].copy()
all_mem_df["img_path"] = all_mem_df["img_path"].apply(lambda x: str(x)[:-4] + ".png")

print(
    f"demographics={demographics_df.participant.nunique()}, "
    f"day1={day1_df.participant.nunique()}, day2={day2_df.participant.nunique()}, "
    f"images={len(all_mem_df)}"
)
demographics_df.head()


## Build `learn_df` (Day 1 learning trials)


In [ ]:
learn_df = day1_df[
    [
        "participant",
        "trials_2.thisRepN",
        "block",
        "prob",
        "MB1_draw_key.keys",
        "MB1_draw_key.rt",
        "reward",
        "img_path",
        "Gender",
        "trials_3.thisRepN",
    ]
].copy()

# PsychoPy: drop memory-loop rows; keep reward-present rows; rebuild 1..80 trial index
learn_df = learn_df[np.isnan(learn_df["trials_3.thisRepN"])].reset_index(drop=True)
trials = (
    learn_df[~np.isnan(learn_df["trials_2.thisRepN"])]["trials_2.thisRepN"].astype(int) + 1
).tolist()
learn_df = learn_df[~np.isnan(learn_df["reward"])].reset_index(drop=True)
learn_df = (
    learn_df[np.isnan(learn_df["trials_2.thisRepN"])]
    .dropna(subset=["Gender"])
    .reset_index(drop=True)
)
learn_df["learn_trials"] = trials
learn_df = learn_df.drop(columns=["trials_2.thisRepN", "trials_3.thisRepN"])
learn_df = learn_df.rename(
    columns={"MB1_draw_key.keys": "gamble", "MB1_draw_key.rt": "gamble_rt"}
)
learn_df["reward"] = learn_df["reward"] / 100
# gamble: PsychoPy key 1.0 → male (0), else female (1)
learn_df["gamble"] = learn_df["gamble"].apply(lambda x: 0 if x == 1.0 else 1)
learn_df["male"] = (learn_df["gamble"] == 0).astype(int)
learn_df["female"] = (learn_df["gamble"] == 1).astype(int)

good_map = flag_good_dm_from_learn(learn_df, REWARD_THRESHOLD)
learn_df = attach_good_dm_flag(learn_df, good_map)

n_learn = learn_df.participant.nunique()
n_good = int(good_map.sum())
n_bad = int((~good_map).sum())
counts = learn_df.groupby("participant").size()
assert counts.eq(NTRIALS).all(), f"expected {NTRIALS} trials/subject; got {counts.value_counts().to_dict()}"
print(f"learn_df: {n_learn} subjects × {NTRIALS} trials; good_dm={n_good}, poor_dm={n_bad}")
learn_df.head()


## Day 1 immediate recognition → `data_df`


In [ ]:
mem_df = day1_df[
    [
        "participant",
        "trials_3.thisRepN",
        "MEM2_recall_key.keys",
        "MEM2_recall_key.rt",
        "MEM2_conf_slider.rating",
        "MEM2_conf_slider.rt",
        "trialIm",
    ]
].copy()
mem_df = mem_df[~np.isnan(mem_df["trials_3.thisRepN"])].reset_index(drop=True)
mem_df = mem_df.rename(
    columns={
        "trials_3.thisRepN": "mem_trials",
        "trialIm": "img_path",
        "MEM2_recall_key.keys": "recog_choice",
        "MEM2_recall_key.rt": "recog_rt",
        "MEM2_conf_slider.rating": "conf_choice",
        "MEM2_conf_slider.rt": "conf_rt",
    }
)
mem_df["condition"] = "day_1"

data_df = mem_df.merge(
    learn_df, on=["participant", "img_path"], how="left", indicator=True, suffixes=(None, "_learn")
)
data_df = add_recognition_outcomes(data_df)
data_df = compute_recognition_sdt(data_df)
data_df = attach_image_dprime(data_df, all_mem_df)

print(
    f"day_1 memory: {data_df.participant.nunique()} subjects, {len(data_df)} rows "
    f"(~{len(data_df) / data_df.participant.nunique():.0f}/subj)"
)


## Day 2 delayed recognition → `data_df2`


In [ ]:
mem_df2 = (
    day2_df[
        [
            "participant",
            "trials.thisRepN",
            "key_resp.keys",
            "key_resp.rt",
            "new_slider.rt",
            "new_slider.rating",
            "trialIm",
        ]
    ]
    .rename(
        columns={
            "trialIm": "img_path",
            "trials.thisRepN": "mem_trials",
            "key_resp.keys": "recog_choice",
            "key_resp.rt": "recog_rt",
            "new_slider.rating": "conf_choice",
            "new_slider.rt": "conf_rt",
        }
    )
    .dropna(subset=["img_path"])
)
mem_df2["condition"] = "day_2"

data_df2 = mem_df2.merge(
    learn_df, on=["participant", "img_path"], how="left", indicator=True, suffixes=(None, "_learn")
)
data_df2 = add_recognition_outcomes(data_df2)
data_df2 = compute_recognition_sdt(data_df2)
data_df2 = attach_image_dprime(data_df2, all_mem_df)

print(
    f"day_2 memory: {data_df2.participant.nunique()} subjects, {len(data_df2)} rows"
)


## Combine days and attach learn-based `good_dm_performance`

**No row dropping** — keep all participants; downstream notebooks / `learn_df_to_pyem(..., include_subjects="good_dm")` filter via the flag.


In [ ]:
full_data_df = pd.concat([data_df, data_df2], ignore_index=True)
# Drop the learn-side flag if merge brought it in, then re-attach from learn totals only
if "good_dm_performance" in full_data_df.columns:
    full_data_df = full_data_df.drop(columns=["good_dm_performance"])
full_data_df = attach_good_dm_flag(full_data_df, good_map)

# Derived RTs (NaN-safe for lure trials with no gamble_rt)
with np.errstate(invalid="ignore", divide="ignore"):
    full_data_df["gamble_rt_log"] = np.log(full_data_df["gamble_rt"].astype(float))
    full_data_df["recog_rt_log"] = np.log(full_data_df["recog_rt"].astype(float))

n_total = full_data_df.participant.nunique()
n_day1 = full_data_df.loc[full_data_df.condition == "day_1", "participant"].nunique()
n_day2 = full_data_df.loc[full_data_df.condition == "day_2", "participant"].nunique()
n_good_full = full_data_df.loc[full_data_df.good_dm_performance, "participant"].nunique()
n_day1_good = full_data_df.loc[
    full_data_df.good_dm_performance & (full_data_df.condition == "day_1"), "participant"
].nunique()
n_day2_good = full_data_df.loc[
    full_data_df.good_dm_performance & (full_data_df.condition == "day_2"), "participant"
].nunique()

print(f"Total subjects in full_data_df: {n_total}")
print(f"  day_1: {n_day1} (good_dm={n_day1_good})")
print(f"  day_2: {n_day2} (good_dm={n_day2_good})")
print(f"Good DM (any condition): {n_good_full}")


## Cohort Summary & Quality Control Assertions


In [ ]:
# Primary assertions (the reason this notebook exists)
assert n_day1 == 208, f"expected 208 day_1 subjects, got {n_day1}"
assert n_day1_good == 200, f"expected 200 day_1 good_dm, got {n_day1_good}"
assert n_day2 == 166, f"expected 166 day_2 subjects, got {n_day2}"
assert n_day2_good == 161, f"expected 161 day_2 good_dm, got {n_day2_good}"
assert n_good == 200, f"learn good_dm expected 200, got {n_good}"

learn_good = set(learn_df.loc[learn_df.good_dm_performance, "participant"])
mem_day1_good = set(
    full_data_df.loc[
        full_data_df.good_dm_performance & (full_data_df.condition == "day_1"), "participant"
    ]
)
assert learn_good == mem_day1_good, (
    f"learn/memory day_1 good_dm mismatch: "
    f"learn-only={len(learn_good - mem_day1_good)}, mem-only={len(mem_day1_good - learn_good)}"
)

# Recognition memory exclusion (7 additional participants with d' < 0 -> n = 193)
d1_good = full_data_df.loc[full_data_df.good_dm_performance & (full_data_df.condition == "day_1")]
n_day1_mem_good = d1_good.loc[pd.to_numeric(d1_good["dprime"], errors="coerce") >= 0, "participant"].nunique()
print(f"Day-1 recognition memory cohort after d' >= 0 exclusion: n = {n_day1_mem_good}")
assert n_day1_mem_good == 193, f"expected 193 day_1 memory subjects, got {n_day1_mem_good}"


## Save v2 CSVs

Exports preprocessed learning and recognition tables for downstream hierarchical modeling.


In [ ]:
# Columns to keep (explicit, no Unnamed / model params)
learn_cols = [
    "participant",
    "block",
    "prob",
    "gamble",
    "gamble_rt",
    "reward",
    "img_path",
    "Gender",
    "learn_trials",
    "male",
    "female",
    "good_dm_performance",
]
full_cols = [
    "participant",
    "mem_trials",
    "recog_choice",
    "recog_rt",
    "conf_choice",
    "conf_rt",
    "img_path",
    "condition",
    "block",
    "prob",
    "gamble",
    "gamble_rt",
    "reward",
    "Gender",
    "learn_trials",
    "male",
    "female",
    "_merge",
    "recog_choice_bool",
    "hits",
    "misses",
    "false_alarms",
    "correct_rejections",
    "hit_rate",
    "false_alarm_rate",
    "dprime",
    "crit",
    "auc",
    "mcc",
    "image_dprime",
    "good_dm_performance",
    "gamble_rt_log",
    "recog_rt_log",
]

learn_out = learn_df[learn_cols].copy()
full_out = full_data_df[full_cols].copy()

learn_path = SAVE_DIR / "learn_df_prolific_v2.csv"
full_path = SAVE_DIR / "full_data_df_prolific_v2.csv"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
learn_out.to_csv(learn_path, index=False)
full_out.to_csv(full_path, index=False)

# Workspace mirror for offline / sandbox use
ws_data = Path("..") / "data"
ws_data.mkdir(parents=True, exist_ok=True)
learn_out.to_csv(ws_data / "learn_df_prolific_v2.csv", index=False)
full_out.to_csv(ws_data / "full_data_df_prolific_v2.csv", index=False)

print("wrote", learn_path, f"({learn_out.participant.nunique()} subjects, {len(learn_out)} rows)")
print("wrote", full_path, f"({full_out.participant.nunique()} subjects, {len(full_out)} rows)")
print("mirrored to", ws_data.resolve())
print(full_out.groupby("condition")["participant"].nunique().to_dict())
print(
    "good_dm by condition:",
    full_out.loc[full_out.good_dm_performance]
    .groupby("condition")["participant"]
    .nunique()
    .to_dict(),
)


## Notes for downstream model fitting

1. Load `learn_df_prolific_v2.csv` into `learn_df_to_pyem(..., include_subjects="good_dm")` → 200 subjects.
2. Load `full_data_df_prolific_v2.csv`, rename `learn_trials`→`trials_gamble`, `block`→`block_gamble`, `reward`→`reward_gamble`, `dprime`→ subject d′ column as needed, then optionally filter `good_dm_performance`.
3. After hierarchical RW fit, attach trial RPE / subject α,β with `attach_rw_rpe_new` / `attach_model_subject_params`.
4. Day-1 `glmer` should then see ~200 subjects; Day-2 ~161.
